## create_sql_query_chain
### create_sql_query_chain读取数据库的表结构信息
### 把这些结构信息连同你的问题一起发给大模型
### 让大模型生成一条合适的 SQL


In [6]:
from langchain_classic.chains import create_sql_query_chain
from langchain_community.utilities import SQLDatabase
import os
import dotenv

dotenv.load_dotenv()
#测试本地数据库的连接
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST") #或者127。0.0.1
db_port = os.getenv("DB_PORT")
db_database = "performance_schema"
#mysql+pymysql://用户名：密码@ip地址:端口/数据库名
db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_database}")

print(f"操作的是哪种数据库{db.dialect}")
print(f"数据库中的表有{db.get_usable_table_names()}")

#执行查询操作
res=db.run("SELECT COUNT(*) FROM accounts")
print(res)


操作的是哪种数据库mysql
数据库中的表有['accounts', 'binary_log_transaction_compression_stats', 'cond_instances', 'data_lock_waits', 'data_locks', 'error_log', 'events_errors_summary_by_account_by_error', 'events_errors_summary_by_host_by_error', 'events_errors_summary_by_thread_by_error', 'events_errors_summary_by_user_by_error', 'events_errors_summary_global_by_error', 'events_stages_current', 'events_stages_history', 'events_stages_history_long', 'events_stages_summary_by_account_by_event_name', 'events_stages_summary_by_host_by_event_name', 'events_stages_summary_by_thread_by_event_name', 'events_stages_summary_by_user_by_event_name', 'events_stages_summary_global_by_event_name', 'events_statements_current', 'events_statements_histogram_by_digest', 'events_statements_histogram_global', 'events_statements_history', 'events_statements_history_long', 'events_statements_summary_by_account_by_event_name', 'events_statements_summary_by_digest', 'events_statements_summary_by_host_by_event_name', 'events_s

### 举例2 chain的使用
create_sql_query_chain读取数据库的表结构信息
把这些结构信息连同你的问题一起发给大模型
让大模型生成一条合适的 SQL

In [10]:
import os
from langchain_classic.chains import create_sql_query_chain
from langchain_community.utilities import SQLDatabase
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
import dotenv
dotenv.load_dotenv()


#测试本地数据库的连接
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST") #或者127。0.0.1
db_port = os.getenv("DB_PORT")
db_database = "performance_schema"
#mysql+pymysql://用户名：密码@ip地址:端口/数据库名
db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_database}")


llm = ChatOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    temperature=0,
    model="deepseek-chat",
)
#创建一个实例
chain = create_sql_query_chain(llm, db)
response=chain.invoke({"question":"数据表accounts有多少行？"})###只能叫question
print(response)

SQLQuery: SELECT COUNT(*) AS `row_count` FROM `accounts`;


# create_stuff_documents_chain

In [11]:
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "What are everyone's favorite colors:\n\n{context}")
    ]
)

chain = create_stuff_documents_chain(llm, prompt)
# chain = create_stuff_documents_chain(llm, prompt,document_variable_name="context")
docs_data = [
    Document(
        page_content="Jesse loves red but not yellow"
    ),
    Document(
        page_content="Jamal loves green but not as much as he loves orange"
    ),
]

response=chain.invoke({"context": docs_data})
print(response)

Let’s break this down step by step.

---

**1. Jesse’s favorite colors**  
> Jesse loves red but not yellow.  
This means red is a favorite, yellow is not.  
We don’t know if Jesse loves other colors besides red.

---

**2. Jamal’s favorite colors**  
> Jamal loves green but not as much as he loves orange.  
This means Jamal loves green, but orange is his *top* favorite (or at least more than green).  
So Jamal definitely loves orange and green.

---

**3. Summary of favorites so far**  
- **Jesse**: red ✅, yellow ❌, others unknown.  
- **Jamal**: orange ✅, green ✅, others unknown.

---

**4. What the question is asking**  
The question says:  
> What are everyone's favorite colors:  
> Jesse loves red but not yellow  
> Jamal loves green but not as much as he loves orange  

It seems like we’re supposed to list each person’s favorite colors based only on the given info.

---

**5. Possible interpretation**  
Sometimes these puzzles assume each person has *one* favorite color, but here